# Quantum Kernel Feasibility Test — Ensemble QK-LSTM
**Purpose:** Confirm that the quantum kernel component of the QK-LSTM model runs in Google Colab using a classical state-vector simulator, with no physical quantum hardware.

This directly addresses **Item 9, bullet 10** of the Outline Oral Defense Minutes: *"To check the simulation of Colab if quantum is applicable."*

The circuit below matches the design described in the manuscript (Section 2.6.2.1):
- 4 qubits
- Hadamard superposition
- Ry(θ)·Rz(φ) data encoding
- CNOT entanglement chain
- Kernel = |⟨φ(vⱼ)|φ(vₜ)⟩|² measured as the probability of returning to the all-zero state


## Step 1 — Install PennyLane
Run this once per Colab session. (Qiskit would also work; PennyLane is used here because it integrates directly with PyTorch/TensorFlow for the LSTM side.)

In [ ]:
!pip install --quiet pennylane

## Step 2 — Build the 4-qubit quantum feature map and kernel
This defines U(v) = U_ent · U_enc exactly as written in the manuscript.

In [ ]:
import pennylane as qml
from pennylane import numpy as np

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)   # classical state-vector simulator

def feature_map(v):
    """Encodes a classical vector v into a quantum state |phi(v)>.
    Mirrors U(v) = U_ent . U_enc from the paper."""
    for k in range(n_qubits):
        qml.Hadamard(wires=k)              # superposition (|psi0>)
    for k in range(n_qubits):
        qml.RY(v[2 * k],     wires=k)      # theta_k  (Ry rotation)
        qml.RZ(v[2 * k + 1], wires=k)      # phi_k    (Rz rotation)
    for k in range(n_qubits - 1):
        qml.CNOT(wires=[k, k + 1])         # CNOT entanglement chain

@qml.qnode(dev)
def kernel_circuit(v_t, v_j):
    """Applies U(v_t) then U(v_j)^dagger, then measures the all-zero probability."""
    feature_map(v_t)
    qml.adjoint(feature_map)(v_j)
    return qml.probs(wires=range(n_qubits))

def quantum_kernel(v_t, v_j):
    """Quantum kernel value k(v_t, v_j) = |<phi(v_j)|phi(v_t)>|^2."""
    return kernel_circuit(v_t, v_j)[0]     # probability of the |0000> state

## Step 3 — Run a feasibility test
Each input vector has 2 × n_qubits = 8 entries (a θ/φ pair per qubit). In the full model these come from the concatenated [hₜ₋₁ ; xₜ] vector scaled to rotation angles; here we use small sample values just to confirm the circuit executes and behaves correctly.

In [ ]:
# Sample inputs shaped like the model's encoded vectors
v_t   = np.array([0.12, 0.55, 0.31, 0.88, 0.07, 0.42, 0.65, 0.19])
v_j   = np.array([0.15, 0.50, 0.29, 0.91, 0.10, 0.40, 0.60, 0.22])  # similar to v_t
v_far = np.array([2.90, 1.10, 2.50, 0.30, 3.00, 1.80, 0.50, 2.70])  # very different

k_self = quantum_kernel(v_t, v_t)
k_near = quantum_kernel(v_t, v_j)
k_far  = quantum_kernel(v_t, v_far)

print(f"k(v_t, v_t)   = {k_self:.6f}   (identical  -> expect 1.0)")
print(f"k(v_t, v_j)   = {k_near:.6f}   (similar    -> expect high)")
print(f"k(v_t, v_far) = {k_far:.6f}   (different  -> expect low)")
print()
if abs(k_self - 1.0) < 1e-9 and k_near > k_far:
    print("FEASIBILITY CONFIRMED: quantum kernel runs correctly in Colab "
          "on a state-vector simulator, no physical quantum hardware required.")
else:
    print("Unexpected result - re-check the circuit.")

## Step 4 — (Optional) Draw the circuit
Useful as a figure for your defense slides or appendix.

In [ ]:
print(qml.draw(kernel_circuit)(v_t, v_j))

---
### What this shows
- The kernel returns **exactly 1.0** for identical inputs, a **high** value for similar inputs, and a **low** value for very different ones — the defining behavior of a valid similarity kernel.
- It runs entirely on `default.qubit`, a **classical state-vector simulator**, so **no physical quantum machine is needed**.
- Execution is near-instant for 4 qubits and fits in **free Google Colab**.

You can now honestly add the feasibility sentence to the manuscript (Section 2.6 or 2.8).